# Insurance Premium Analysis

Comprehensive analysis of features, target distribution, and model performance.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import joblib

# Set style for better visualization
plt.style.use('seaborn')
sns.set_palette('husl')

## 1. Data Loading and Overview

In [ ]:
# Load all datasets
data_dir = Path('/home/USER/Projects/ikaggle')
train_data = pd.read_csv(data_dir / 'data/train.csv')
validate_data = pd.read_csv(data_dir / 'data/validate.csv')
binary_data = pd.read_csv(data_dir / 'AAA/bitbins/train_bitbins.csv')

print("Dataset shapes:")
print(f"Train:       {train_data.shape}")
print(f"Validate:    {validate_data.shape}")
print(f"Binary:      {binary_data.shape}")


## 2. Target Variable Analysis

In [ ]:
def plot_target_distribution(data, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Original distribution
    sns.histplot(data['PremiumAmount'], bins=50, ax=ax1)
    ax1.set_title(f'{title} - Original Distribution')
    ax1.set_xlabel('PremiumAmount')
    
    # Log-transformed distribution
    sns.histplot(np.log1p(data['PremiumAmount']), bins=50, ax=ax2)
    ax2.set_title(f'{title} - Log-transformed Distribution')
    ax2.set_xlabel('log(PremiumAmount + 1)')
    
    plt.tight_layout()
    return fig

# Plot training data distribution
plot_target_distribution(train_data, 'Training Data')

# Plot validation data distribution
plot_target_distribution(validate_data, 'Validation Data')

## 3. Feature Analysis

In [ ]:
def analyze_features(data):
    features = [col for col in data.columns if col not in ['id', 'PremiumAmount']]
    
    # Calculate correlations with target
    correlations = []
    for feature in features:
        corr = stats.spearmanr(data[feature], data['PremiumAmount'])[0]
        correlations.append({
            'feature': feature,
            'correlation': corr,
            'null_count': data[feature].isnull().sum(),
            'unique_values': data[feature].nunique()
        })
    
    return pd.DataFrame(correlations).sort_values('correlation', key=abs, ascending=False)

# Analyze original features
orig_analysis = analyze_features(train_data)
print("Top 10 Original Features by Correlation:")
display(orig_analysis.head(10))

# Analyze binary features
binary_analysis = analyze_features(binary_data)
print("\nTop 10 Binary Features by Correlation:")
display(binary_analysis.head(10))

## 4. Submission Analysis

In [ ]:
def analyze_submissions(data):
    submission_cols = [col for col in data.columns if col.startswith('submission_')]
    
    # Calculate RMSLE for each submission
    scores = []
    for col in submission_cols:
        rmsle = np.sqrt(np.mean(np.square(
            np.log1p(data[col]) - np.log1p(data['PremiumAmount'])
        )))
        scores.append({
            'submission': col,
            'rmsle': rmsle
        })
    
    # Plot correlation heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        data[submission_cols].corr(),
        annot=True,
        cmap='coolwarm',
        center=0
    )
    plt.title('Submission Correlations')
    plt.tight_layout()
    
    return pd.DataFrame(scores).sort_values('rmsle')

submission_analysis = analyze_submissions(submissions_data)
print("Submission Performance:")
display(submission_analysis)

## 5. Error Analysis

In [ ]:
def analyze_errors(predictions, actuals, features):
    errors = np.abs(np.log1p(predictions) - np.log1p(actuals))
    error_df = pd.DataFrame({
        'actual': actuals,
        'predicted': predictions,
        'error': errors
    })
    
    # Plot error distribution
    plt.figure(figsize=(10, 5))
    sns.histplot(errors, bins=50)
    plt.title('Error Distribution')
    plt.xlabel('|log(actual) - log(predicted)|')
    plt.show()
    
    # Plot errors vs predictions
    plt.figure(figsize=(10, 5))
    plt.scatter(predictions, errors, alpha=0.1)
    plt.title('Errors vs Predictions')
    plt.xlabel('Predicted Value')
    plt.ylabel('Error')
    plt.show()
    
    return error_df.sort_values('error', ascending=False).head(10)

# Load best model predictions
predictions = pd.read_csv(data_dir / 'AAA/predictions/ensemble_predictions.csv')
print("Top 10 Largest Errors:")
display(analyze_errors(
    predictions['Premium Amount'],
    train_data['PremiumAmount'],
    train_data
))